# Calculate mortality at each grid point using central estimates only

$M(x, y) = POP(x, y) \; \times \; BMR_c \; \times \; AF(x, y)  $

This script calculates gridpoint mortality for each mortality health outcome. The final step calculates the sum of the 17 outcomes to estimate total temperature mortality

In [1]:
import os
import glob
import json
import pandas as pd
import numpy as np
import xarray as xr
from utils.mortality_utils import mortality
from utils.utils import get_scenario_config, adjust_longitude, standardise_latlon
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")
TMREL_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "TMREL" / "summaries")

In [3]:
# Load country id mask
mask_file = "country_id_mask_0.25.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataset(mask_path)["country_id"]

# Load population file
pop_file = "ssp2_era5_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

# Load look up file
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

# Load mean tmrel at all gridcells
tmrel_map = xr.open_dataarray(TMREL_DIR/"tmrel_mean_0.25_map.nc")

In [4]:
# === Health variables ===
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [5]:
# Helpers: turn values into integer positions in a lookup axis ---
def nearest_idx(index: pd.Index, vals) -> np.ndarray:
    """Position of nearest value in `index` (must be sorted); -1 where NaN."""
    vals = np.asarray(vals)
    out = np.full(vals.shape, -1, dtype=np.int32)
    ok = np.isfinite(vals)
    grid = np.asarray(index.values, dtype=np.float64)
    steps = np.diff(grid)
    if len(grid) > 1 and np.allclose(steps, steps[0], rtol=1e-6, atol=0):
        # evenly spaced axis: compute the position directly (fast)
        i = np.rint((vals[ok] - grid[0]) / steps[0])
        out[ok] = np.clip(i, 0, len(grid) - 1)
    else:
        # fallback: your original method (slow)
        out[ok] = index.get_indexer(vals[ok], method="nearest")
    return out


def exact_idx(index: pd.Index, vals) -> np.ndarray:
    """Position of exact match in `index`; -1 where NaN or not found."""
    vals = np.asarray(vals)
    out = np.full(vals.shape, -1, dtype=np.int32)
    ok = np.isfinite(vals)
    out[ok] = index.get_indexer(vals[ok].astype(index.dtype))
    return out

In [6]:
def find_logRR(logRR, masks, temp, annual):
    # temp: (time, lat, lon)   annual: (lat, lon)   masks: (lat, lon)
    c = exact_idx(logRR.indexes["country"], masks.values)               # 2D
    a = nearest_idx(logRR.indexes["annual_temperature"], annual.values) # 2D
    d = nearest_idx(logRR.indexes["daily_temperature"], temp.values)    # 3D

    table = logRR.transpose("country", "annual_temperature", "daily_temperature").values

    valid = (c >= 0)[None] & (a >= 0)[None] & (d >= 0)
    vals = table[np.maximum(c, 0)[None], np.maximum(a, 0)[None], np.maximum(d, 0)]
    vals = np.where(valid, vals, np.nan)   # NaN for ocean / unknown country / NaN temps

    return xr.DataArray(vals, coords=temp.coords, dims=temp.dims)

In [7]:
def process_mortality(temp, tmrel, mortality):
    """ Split non-optimal temperature to hot and cold, group by month to save """

    # Create mask for cold related and heat related deaths
    cold = xr.where(temp<tmrel, 1, 0)
    hot = xr.where(temp>tmrel, 1, 0)
    # select mortality, nan where cold/hot is 0
    cold_deaths = mortality.where(cold==1).drop_vars("annual_temperature")
    heat_deaths = mortality.where(hot==1).drop_vars("annual_temperature")

    # Group by month for saving
    cold_month = cold_deaths.groupby("time.month").sum("time")
    heat_month = heat_deaths.groupby("time.month").sum("time")

    return cold_month, heat_month

In [8]:
def add_attrs(da, high_or_low):
    description = (f"Total {health_VAR} mortality due to {high_or_low} temperature using "
                   "central estimates only - scripts by A.F. Wells (2025)")
    da.attrs["description"] = description
    da.attrs["GBD version"] = GBD_version
    da.attrs["health_var"] = health_VAR
    da.attrs["year"] = year
    da.attrs["ensemble_number"] = ens_num
    da.attrs["scenario"] = scenario
    da.attrs["model"] = model
    return da

In [10]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_LE"
# For RR curves and file name
GBD_version = "GBD23"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

# TEMP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "temperature")  # CHANGE WHEN THE DATA IS FIXED
TEMP_DIR = "/glade/derecho/scratch/arielm/Olympics/ARISE-SAI-1.5-downscaled/TREFHT/SSP2-4.5"
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "temperature" / "gridpoint_mortality")

# === Main loop ===
for ens_num in ensemble_members:
    for year in years:
        print(f"Processing {scenario}, Ensemble {ens_num:02d}, Year {year}")

        temp_file = f"bc_TREFHT_{scenario[:-3]}_ens0{ens_num:02d}_{year}.nc"
        temp_path = os.path.join(TEMP_DIR, temp_file)
        # Convert longitude from 0-360 to -180-180
        temp = adjust_longitude(standardise_latlon(xr.open_dataarray(temp_path))) - 273.15  # Kelvin to Celcius

        del temp_file, temp_path

        annual = temp.mean("time")        # (lat, lon), assumes one year per file

        # Find tmrel based on annual mean temperature at each grid cell
        tmrel_everywhere = tmrel_map.sel(annual_temperature=annual, method="nearest")

        for health_VAR in health_vars:
            print(f"Processing mortality outcome {health_VAR}")

            bmr_file = f"{GBD_version}_BMR_Global_Country_Map_era5_{health_VAR}_1990-2009.nc"
            bmr_path = os.path.join(BMR_DIR, bmr_file)
            BMR = xr.open_dataset(bmr_path)["BMR"]  # central estimate

            del bmr_file, bmr_path

            RR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / GBD_version / "RR_curves" / "temperature" / health_VAR)
            # load mean logRR
            logRR_file = f"{health_VAR}_norm_mean_logRR.nc"
            logRR_path = os.path.join(RR_DIR, logRR_file)
            logRR = xr.open_dataarray(logRR_path)

            del RR_DIR, logRR_file, logRR_path

            # Find RR at each grid point
            logRR_exposure = find_logRR(logRR, masks, temp, annual)
            RR = np.exp(logRR_exposure)

            del logRR_exposure, logRR

            # Calculate the attributable fraction
            AF = 1 - (1/RR)

            del RR

            POP = pop.sel(year=year)

            # Calculate mortality at each grid point
            mortality_day = mortality(AF, BMR, POP)

            del AF, BMR, POP

            cold_mortality, heat_mortality = process_mortality(temp, tmrel_everywhere, mortality_day)

            del mortality_day

            cold_mortality = add_attrs(cold_mortality, "low")
            heat_mortality = add_attrs(heat_mortality, "high")

            cold_out_file = f"Monthly_cold_mortality_{GBD_version}_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            cold_out_path = os.path.join(SAVE_DIR, cold_out_file)
            print(f"Saving monthly mortality timeseries to {cold_out_path}")
            cold_mortality.to_netcdf(cold_out_path)
            cold_mortality.close()

            heat_out_file = f"Monthly_heat_mortality_{GBD_version}_{health_VAR}_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            heat_out_path = os.path.join(SAVE_DIR, heat_out_file)
            print(f"Saving monthly mortality timeseries to {heat_out_path}")
            heat_mortality.to_netcdf(heat_out_path)
            heat_mortality.close()

            del cold_mortality, heat_mortality

    del temp, annual

import resource
print(f"Peak memory: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

print("All processing complete.")

Processing SSP245_LE, Ensemble 01, Year 2050
Processing mortality outcome CHRONIC_KIDNEY_DISEASE
Find logRR at each gridpoint
Calculate mortality
Saving monthly mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CHRONIC_KIDNEY_DISEASE_CESM2_SSP245_LE_01_2050.nc
Saving monthly mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_heat_mortality_GBD23_CHRONIC_KIDNEY_DISEASE_CESM2_SSP245_LE_01_2050.nc
Processing mortality outcome CARDIOMYOPATHY_MYOCARDITIS
Find logRR at each gridpoint
Calculate mortality
Saving monthly mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CARDIOMYOPATHY_MYOCARDITIS_CESM2_SSP245_LE_01_2050.nc
Saving monthly mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_heat_mortality_GBD23_CARDIOMYOPATHY_MYOC

## Save the sum of all mortality outcomes

In [11]:
for temp in ["cold", "heat"]:
    for ens_num in ensemble_members:
        for year in years:
            # Find all files for this ensemble
            in_files = f"Monthly_{temp}_mortality_{GBD_version}_*_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            in_path = os.path.join(SAVE_DIR, in_files)
            files = sorted(glob.glob(in_path))

            # Open and combine
            datasets = [xr.open_dataarray(f) for f in files]

            # Align (important in case of slight coordinate mismatches)
            aligned = xr.align(*datasets, join="exact")

            # Sum across the health variables
            summed_da = sum(aligned)

            description = (f"Total mortality due to {temp} temperature using "
                           "central estimates only - scripts by A.F. Wells (2025)")
            summed_da.attrs["description"] = description
            summed_da.attrs["GBD version"] = GBD_version
            summed_da.attrs["year"] = year
            summed_da.attrs["ensemble_number"] = ens_num
            summed_da.attrs["scenario"] = scenario
            summed_da.attrs["model"] = model

            out_file = f"Monthly_{temp}_mortality_{GBD_version}_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving summed mortality timeseries to {out_path}")
            summed_da.to_netcdf(out_path)

print("All processing complete.")

Saving summed mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CESM2_SSP245_LE_01_2050.nc
Saving summed mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CESM2_SSP245_LE_02_2050.nc
Saving summed mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CESM2_SSP245_LE_03_2050.nc
Saving summed mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CESM2_SSP245_LE_04_2050.nc
Saving summed mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CESM2_SSP245_LE_05_2050.nc
Saving summed mortality timeseries to /glade/work/awells/workflow/CESM2/mortality/temperature/gridpoint_mortality/Monthly_cold_mortality_GBD23_CESM2_S